In [ ]:
import ctypes, sys
from pathlib import Path

# Fix libnvrtc on WSL2 (same as training notebook)
NVRTC_LIB = Path(sys.prefix) / 'lib/python3.12/site-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0'
if NVRTC_LIB.exists():
    ctypes.CDLL(str(NVRTC_LIB))
    print(f'Preloaded: {NVRTC_LIB}')

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

PROJECT_ROOT = Path(r"/mnt/c/Users/test/OneDrive - Braude College of Engineering/Software Engineering Stuff/Capstone/Bug-bite-classification-using-a-TC-classification-model")

# Model weights — prefer the saved copy in Model_Weights/, fall back to the run directory
WEIGHTS = PROJECT_ROOT / 'Model_Weights' / 'yolov8_bug_bite_best.pt'
if not WEIGHTS.exists():
    WEIGHTS = PROJECT_ROOT / 'yolo_runs' / 'bug_bite_yolov8' / 'weights' / 'best.pt'

print(f'Using weights: {WEIGHTS}')
print(f'Exists: {WEIGHTS.exists()}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
model = YOLO(str(WEIGHTS))
print(f'Model loaded. Classes: {model.names}')

## Single image inference

In [ ]:
import random

VAL_DIR = PROJECT_ROOT / 'Yolo_Bug_Data' / 'dataset' / 'images' / 'val'
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}
val_images = [p for p in VAL_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS]
test_images = random.sample(val_images, 2)

CONF_THRESHOLD = 0.3
IOU_THRESHOLD  = 0.45

results = model.predict(
    source=[str(p) for p in test_images],
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    device=0,
    verbose=False,
)

for r in results:
    print(f'Image: {Path(r.path).name}  —  {len(r.boxes)} detection(s)')
    for box in r.boxes:
        cls  = model.names[int(box.cls)]
        conf = float(box.conf)
        xyxy = box.xyxy[0].tolist()
        print(f'  {cls}: {conf:.2%}  bbox={[round(v) for v in xyxy]}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, r in zip(axes, results):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f'{Path(r.path).name}  ({len(r.boxes)} det)', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Batch inference on a folder

In [ ]:
# Run on all validation images and show a grid of the first 9 with detections
VAL_DIR = PROJECT_ROOT / 'Yolo_Bug_Data' / 'dataset' / 'images' / 'val'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}
val_images = [p for p in sorted(VAL_DIR.iterdir()) if p.suffix.lower() in IMAGE_EXTS]
print(f'Val images: {len(val_images)}')

batch_results = model.predict(
    source=str(VAL_DIR),
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    device=0,
    stream=True,   # memory-efficient for large folders
    verbose=False,
)

detections_found = []
for r in batch_results:
    if len(r.boxes) > 0:
        detections_found.append(r)

print(f'Images with detections: {len(detections_found)} / {len(val_images)}')

In [ ]:
# Show grid of first 9 images that had detections
sample = detections_found[:9]
n = len(sample)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, r in zip(axes, sample):
    annotated = r.plot()
    ax.imshow(annotated[:, :, ::-1])
    ax.set_title(Path(r.path).name, fontsize=8)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()